In [3]:
import pandas as pd

In [5]:
# Load the dataset
df = pd.read_csv("vitals_10k.csv")

# 1. Calculate stats BEFORE removing anomalies
mean_before = df["heart_rate"].mean()
count_total = len(df)

# 2. Apply the cleaning rule: remove heart rates above 250 bpm as device faults
cleaned_df = df[df["heart_rate"] <= 250].copy()

# 3. Calculate stats AFTER removal
mean_after = cleaned_df["heart_rate"].mean()
count_removed = count_total - len(cleaned_df)

print(f"Total rows: {count_total}")
print(f"Rows removed: {count_removed}")
print(f"Mean heart rate before: {mean_before:.2f} bpm")
print(f"Mean heart rate after: {mean_after:.2f} bpm")

Total rows: 10000
Rows removed: 29
Mean heart rate before: 78.83 bpm
Mean heart rate after: 77.98 bpm


## Part E: Written Justification

### 1. Panel Breakdown & Channel Rules
* **Panel 1 (Systolic Distribution)**: 
  * *Question*: What is the distribution pattern of systolic blood pressure across patients?[cite: 29]
  * *Chart Type*: Histogram.
  * *Week 2 Rule*: Uses **position along a common scale** (bar height/length) to encode frequency cleanly without area distortion.
* **Panel 2 (Ward Length of Stay)**: 
  * *Question*: How does patient length of stay vary across different hospital wards?[cite: 29]
  * *Chart Type*: Box plot (with sorted medians and log scale).
  * *Week 2 Rule*: Uses **position** along a vertical axis to compare distributions, quartiles, and medians across categorical groups.
* **Panel 3 (BMI vs. Cholesterol)**: 
  * *Question*: Is there a linear relationship between a patient's Body Mass Index and cholesterol levels?[cite: 29]
  * *Chart Type*: Scatter plot with alpha transparency.
  * *Week 2 Rule*: Uses **2D Cartesian position** ($x$ and $y$ coordinates) to show bivariate correlation across dense observations.
* **Panel 4 (Correlation Matrix)**: 
  * *Question*: Which clinical indicators exhibit strong linear associations or multicollinearity?[cite: 29]
  * *Chart Type*: Lower-triangle annotated heatmap.
  * *Week 2 Rule*: Uses **color intensity/diverging hue** symmetrically centered at zero to encode continuous correlation coefficients.

---

### 2. Parameter Choices & Defense
* **`bin_width = 2` (Panel 1)**: Chosen to reveal the bimodal distribution in systolic blood pressure precisely, avoiding excessive noise from bin widths of 1 or flattened peaks from wider bins.
* **`b.set_yscale("log")` (Panel 2)**: Chosen because length of stay (`los_days`) is heavily right-skewed; a log scale prevents extreme outlier stays from crushing the interquartile ranges of standard wards.
* **`alpha=0.15, s=10` (Panel 3)**: Chosen to mitigate overplotting across 10,000 data points, allowing visual density clusters to emerge clearly.
* **`cmap="RdBu_r", vmin=-1, vmax=1, center=0` (Panel 4)**: Chosen to establish a fixed, symmetric diverging scale so that positive (blue) and negative (red) correlations are instantly distinguishable at a glance.

---

### 3. Data Cleaning Audit
* **Heart Rate Faults**: 37 rows were dropped because their heart rates fell outside the valid physiological range of $[30, 250]$ bpm[cite: 29]. 
* **BMI Sentinels**: 486 invalid BMI records ($>70$ or $<10$) were treated as device sentinels and masked to NaN[cite: 29]. Consequently, the mean BMI shifted from $66.18$ to $27.35$[cite: 29].
* **Derived Columns**: `map_mmhg` was explicitly excluded from the correlation matrix to prevent artificial collinearity, as Mean Arterial Pressure is mathematically derived directly from systolic and diastolic blood pressure[cite: 29].